IPO Performance Analysis and Listing Gain Prediction Using Machine Learning

Objective:
To analyze historical IPO performance and identify factors affecting listing gains using statistical analysis and machine learning techniques.

Tools:
Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn

Dataset:
561 IPOs

Machine Learning Models:
1. Linear Regression
2. Random Forest Regressor

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [ ]:
df = pd.read_excel("Initial Public Offering.xlsx")

print("Shape:", df.shape)
df.head()

In [ ]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

print(df.columns.tolist())

In [ ]:
df.columns = [
    'date',
    'ipo_name',
    'issue_size_crores',
    'qib',
    'hni',
    'rii',
    'total_subscription',
    'offer_price',
    'list_price',
    'listing_gain',
    'cmp_bse',
    'cmp_nse',
    'current_gains'
]

In [ ]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
print("Total IPOs:", len(df))

print("Average Listing Gain:",
      round(df['listing_gain'].mean(),2))

print("Median Listing Gain:",
      round(df['listing_gain'].median(),2))

print("Highest Listing Gain:",
      round(df['listing_gain'].max(),2))

print("Lowest Listing Gain:",
      round(df['listing_gain'].min(),2))

In [ ]:
top10 = df.sort_values(
    by='listing_gain',
    ascending=False
).head(10)

top10[['ipo_name','listing_gain']]

In [ ]:
worst10 = df.sort_values(
    by='listing_gain'
).head(10)

worst10[['ipo_name','listing_gain']]

In [ ]:
corr = df[
    [
        'issue_size_crores',
        'qib',
        'hni',
        'rii',
        'total_subscription',
        'offer_price',
        'list_price',
        'listing_gain',
        'current_gains'
    ]
].corr()

print(
    corr['listing_gain']
    .sort_values(ascending=False)
)

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    df['listing_gain'],
    bins=25
)

plt.title("Distribution of Listing Gain")

plt.xlabel("Listing Gain %")

plt.ylabel("Count")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    df['total_subscription'],
    df['listing_gain']
)

plt.xlabel("Total Subscription")

plt.ylabel("Listing Gain")

plt.title("Subscription vs Listing Gain")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    df['qib'],
    df['listing_gain']
)

plt.xlabel("QIB")

plt.ylabel("Listing Gain")

plt.title("QIB vs Listing Gain")

plt.show()

In [ ]:
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year

yearly = (
    df.groupby('year')
      ['listing_gain']
      .mean()
)

yearly.plot(
    kind='line',
    marker='o',
    figsize=(8,5)
)

plt.title(
    "Average Listing Gain by Year"
)

plt.show()

In [ ]:
features = [
    'issue_size_crores',
    'qib',
    'hni',
    'rii',
    'total_subscription',
    'offer_price'
]

ml_df = df[
    features +
    ['listing_gain']
].dropna()

X = ml_df[features]

y = ml_df['listing_gain']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
lr = LinearRegression()

lr.fit(
    X_train,
    y_train
)

lr_pred = lr.predict(X_test)

print("Linear Regression")

print(
    "R²:",
    round(
        r2_score(
            y_test,
            lr_pred
        ),
        3
    )
)

print(
    "MAE:",
    round(
        mean_absolute_error(
            y_test,
            lr_pred
        ),
        2
    )
)

print(
    "RMSE:",
    round(
        np.sqrt(
            mean_squared_error(
                y_test,
                lr_pred
            )
        ),
        2
    )
)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(
    X_train,
    y_train
)

rf_pred = rf.predict(X_test)

print("Random Forest")

print(
    "R²:",
    round(
        r2_score(
            y_test,
            rf_pred
        ),
        3
    )
)

print(
    "MAE:",
    round(
        mean_absolute_error(
            y_test,
            rf_pred
        ),
        2
    )
)

print(
    "RMSE:",
    round(
        np.sqrt(
            mean_squared_error(
                y_test,
                rf_pred
            )
        ),
        2
    )
)

In [ ]:
importance = pd.DataFrame({
    'Feature': features,
    'Importance':
    rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    y_test,
    rf_pred
)

plt.xlabel(
    "Actual Listing Gain"
)

plt.ylabel(
    "Predicted Listing Gain"
)

plt.title(
    "Actual vs Predicted"
)

plt.show()

In [ ]:
print("""
PROJECT CONCLUSION

1. Total Subscription showed the strongest relationship with listing gains.

2. QIB and HNI participation were important indicators of IPO success.

3. Investor demand had greater impact than issue size.

4. Machine Learning models were able to predict listing gains with moderate accuracy.

5. Subscription metrics can help investors evaluate IPO opportunities.
""")